### Konteks / Skenario

"Kopi Nusantara" ingin membangun model untuk memprediksi apakah seorang pelanggan akan memberikan **ulasan positif (rating ≥ 4) atau tidak**, berdasarkan atribut pesanan mereka — berguna untuk mengidentifikasi kombinasi menu/cabang yang berisiko mengecewakan pelanggan sebelum benar-benar terjadi.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [5]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas12") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 14:43:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


**A. Membangun ML Pipeline** *(bobot 25%)*

Buat `Pipeline` dengan tahapan: `StringIndexer` untuk `nama_menu`, `cabang`, `waktu_pesan` → `OneHotEncoder` untuk ketiganya → `VectorAssembler` (gabungkan hasil one-hot encoding dengan `qty` dan `waktu_tunggu_menit`) → `LogisticRegression` dengan `labelCol="ulasan_positif"`.

In [6]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Tahap 1-3: StringIndexer untuk setiap kolom kategorikal
idx_kategori = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_kota = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_metode = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# Tahap 4: OneHotEncoder — mengubah ketiga kolom indeks menjadi representasi biner sekaligus
ohe = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"]
)

# Tahap 5: VectorAssembler — menggabungkan seluruh fitur menjadi satu kolom "features"
assembler = VectorAssembler(
    inputCols=["qty", "waktu_tunggu_menit", "nama_menu_vec", "cabang_vec", "waktu_pesan_vec"],
    outputCol="features"
)

# Tahap 6: Algoritma klasifikasi
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

print("Keenam tahap pipeline berhasil didefinisikan (belum dijalankan).")

Keenam tahap pipeline berhasil didefinisikan (belum dijalankan).


**B. Pelatihan & Evaluasi** *(bobot 25%)*

Bagi data 80:20 (`seed=77`), latih pipeline, lalu evaluasi pada data testing menggunakan **accuracy**, **F1-score**, dan **AUC-ROC**. Tampilkan juga *confusion matrix* sederhana menggunakan `groupBy`.


In [9]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)

pipeline = Pipeline(stages=[idx_kategori, idx_kota, idx_metode, ohe, assembler, lr])
train, test = df.randomSplit([0.8, 0.2], seed=77)
model = pipeline.fit(train)

# prediksi
predictions = model.transform(test)

# Evaluasi
ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
accuracy = ev_acc.evaluate(predictions)

ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
f1 = ev_f1.evaluate(predictions)

ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")
auc = ev_auc.evaluate(predictions)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"AUC-ROC  : {auc:.4f}")
print()
predictions.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



**C. Model Pembanding** *(bobot 20%)*

Latih **`RandomForestClassifier`** (`numTrees=30`) menggunakan pipeline dengan lima tahap feature engineering yang identik dari bagian A (hanya tahap terakhir yang diganti). Bandingkan ketiga metrik evaluasinya dengan model Logistic Regression pada bagian B.

In [10]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)
pipeline_rf = Pipeline(stages=[idx_kategori, idx_kota, idx_metode, ohe, assembler, rf])

# Melatih & Prediction
model_rf = pipeline_rf.fit(train)
predictions_rf = model_rf.transform(test)

# Evaluasi
ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
accuracy_rf = ev_acc.evaluate(predictions_rf)

ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
f1_rf = ev_f1.evaluate(predictions_rf)

ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")
auc_rf = ev_auc.evaluate(predictions_rf)

print(f"Accuracy : {accuracy_rf:.4f}")
print(f"F1-score : {f1_rf:.4f}")
print(f"AUC-ROC  : {auc_rf:.4f}")

Accuracy : 0.7980
F1-score : 0.7974
AUC-ROC  : 0.8459


**D. Analisis Kesalahan Model** *(bobot 15%)*

Dari hasil prediksi model **terbaik** (pilih salah satu dari B atau C berdasarkan AUC tertinggi), tampilkan 10 baris di mana model **salah** memprediksi (`label != prediction`). Amati: apakah ada pola tertentu pada kesalahan tersebut (mis. mayoritas terjadi pada `waktu_tunggu_menit` yang berada di sekitar ambang batas tertentu)?


In [21]:
from pyspark.sql.functions import col, when, count, avg, round as spark_round

prediksi_terbaik = predictions if auc >= auc_rf else predictions_rf
nama_terbaik = "Logistic Regression" if auc >= auc_rf else "Random Forest"
print("Model terbaik (AUC tertinggi):", nama_terbaik)

salah = prediksi_terbaik.filter(col("ulasan_positif") != col("prediction"))
print("Jumlah salah prediksi:", salah.count())

salah.select("nama_menu", "cabang", "waktu_pesan", "qty",
             "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability") \
     .show(10, truncate=False)

analisis = (prediksi_terbaik
    .withColumn("salah", (col("ulasan_positif") != col("prediction")).cast("int"))
    .withColumn("rentang",
        when(col("waktu_tunggu_menit") <= 6, "1)   2-6 mnt")
        .when(col("waktu_tunggu_menit") <= 10, "2)  7-10 mnt")
        .when(col("waktu_tunggu_menit") <= 14, "3) 11-14 mnt")
        .when(col("waktu_tunggu_menit") <= 18, "4) 15-18 mnt")
        .otherwise("5) 19-24 mnt"))
    .groupBy("rentang")
    .agg(count("*").alias("total"),
         spark_round(avg("salah"), 3).alias("tingkat_error"))
    .orderBy("rentang"))
analisis.show()

Model terbaik (AUC tertinggi): Logistic Regression
Jumlah salah prediksi: 122
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414356,0.37461208245856437]|
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149019,0.08861774338509809]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008503,0.7679086206099149]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016906074,0.5738454598309393]|
|Americano|Magelang|Sore       |3  |5                 |0     


**E. Kesimpulan Bisnis** *(bobot 15%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan model yang telah dibangun, apa rekomendasi konkret yang dapat diberikan kepada manajemen "Kopi Nusantara" untuk meningkatkan kepuasan pelanggan? Kaitkan dengan fitur yang paling berpengaruh terhadap prediksi (boleh merujuk pada pola yang ditemukan di bagian D).

Dari pola yang ditemukan pada bagian D saya bisa melihat banyaknya total pesanan selama rentang waktu tertentu, saya dapat melihat pola bahwa banyak pelanggan yang mendapat rentang pelayanan dalam rentang waktu 2 hingga 10 menit, terbilang cukup banyak walaupun disisi lain pelayanan kopi nusantara bisa dibilang lambat, karena pelanggan yang mendapatkan pelayanan yang lama juga terbilang tidak sedikit, ada 170 pesanan. Rekomendasi untuk kopi nusantara dari saya yaitu menetapkan target waktu penyajian, dengan waktu penyajian yang sudah di target misalnya dibawah 10 menit, ini akan meningkatkan kepuasan pelanggan. Tambahkan pulan diskon pesanan bila pesanan datang melebihi waktu yang disajikan, jika misal terjadi suatu kejadian terbutuk misalnya penyajian terlambat, penilaian buruk darii pelanggan terhadap gerai dapat diminimalisasi, dengan adanya diskon keterlambatan penyajian pelanggan juga merasa dihargai.